# UD5.03. Keras: el entorno de modelado

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloques 7 y 8 de los apuntes · Criterios **2.b** y **2.c**

---

En el cuaderno anterior escribiste una red entera en NumPy. Aquí se hace lo mismo en
veinte líneas, y la comparación **es** el criterio 2.b: *caracterizar entornos de modelo
de aplicaciones de inteligencia artificial*.

Caracterizar no es describir. Es decir qué aporta cada pieza, qué cuesta, y en qué se
nota. Así que este cuaderno mide: líneas, tiempo, memoria, y qué se gana y qué se pierde.

La segunda mitad es el criterio 2.c: **la ficha de arquitectura**, que es lo que se
rellena antes de escribir la primera capa.

In [ ]:
import os
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

t0 = time.perf_counter()
import keras
t_import = time.perf_counter() - t0

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

keras.utils.set_random_seed(20262027)

print("keras", keras.__version__, "sobre", keras.config.backend())
print(f"Importar Keras ha tardado {t_import:.1f} s.")
print()
print("Esa cifra no es una curiosidad: con un modelo pequeño sobre datos pequeños,")
print("el entrenamiento entero en NumPy del cuaderno anterior acaba antes de que")
print("Keras termine de cargar. Eso NO significa que NumPy sea mejor; significa que")
print("la comparacion hay que hacerla en el regimen en el que se va a trabajar,")
print("y decirlo es parte de caracterizar el entorno.")

---

## 1. Las tres capas del entorno, y el lío de nombres

| Pieza | Qué es |
|---|---|
| **TensorFlow** | el motor: tensores, operaciones, derivación automática, CPU/GPU/TPU |
| **Keras 3** | la API de alto nivel: capas, modelos, entrenamiento, retrollamadas |
| **JAX, PyTorch** | otros motores, que Keras 3 también sabe usar |

Keras nació aparte, se absorbió dentro de TensorFlow como `tf.keras`, y con Keras 3 volvió
a separarse. Eso importa al leer código de internet:

> **`import tensorflow as tf` y luego `tf.keras.layers.Dense` es la forma antigua.**
> La de ahora es `import keras` y `keras.layers.Dense`. Las dos funcionan, y mezclarlas en
> el mismo fichero es la fuente número uno de errores raros.

En esta unidad se escribe siempre `import keras`.

In [ ]:
# Traduccion del material antiguo que te vas a encontrar.
traducciones = [
    ("tf.keras.layers.Dense",
     "keras.layers.Dense"),
    ("tf.keras.models.Sequential",
     "keras.Sequential"),
    ("from tensorflow.keras.preprocessing.image import ImageDataGenerator",
     "keras.layers.RandomFlip / RandomRotation / RandomZoom, dentro del modelo"),
    ("model.save('m.h5')",
     "model.save('m.keras')"),
    ("keras.optimizers.Adam(lr=0.001)",
     "keras.optimizers.Adam(learning_rate=0.001)"),
    ("model.predict_classes(X)",
     "(model.predict(X) >= umbral).astype(int)"),
]

print(f"{'lo que vas a leer por ahi':62} lo que hay que escribir")
print("-" * 120)
for viejo, nuevo in traducciones:
    print(f"{viejo:62} {nuevo}")

---

## 2. La misma red del cuaderno 02, en Keras

Sesenta líneas de NumPy contra esto:

In [ ]:
CARACTERISTICAS = ["recencia_dias", "frecuencia", "monetario", "ticket_medio",
                   "antiguedad_dias", "categorias_distintas", "proporcion_movil",
                   "dias_entre_pedidos", "cat_informatica", "cat_telefonia"]

# Solo en Google Colab: descarga el fichero de datos de la unidad. En local
# ya esta en recursos/datos/ y esta celda no hace nada.
#
# Ojo con la ruta: en el espejo publico la unidad se llama UD5, no
# UD5_tensores_redes_neuronales, porque sincroniza_notebooks.py acorta el nombre.
import urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD5/datos/")
FICHERO = "clientes_abandono.csv"

os.makedirs("datos", exist_ok=True)
RUTA_DATOS = os.path.join("datos", FICHERO)
if not os.path.exists(RUTA_DATOS):
    urllib.request.urlretrieve(BASE + FICHERO, RUTA_DATOS)
    print("descargado:", FICHERO)
else:
    print("ya esta:", FICHERO)

tabla = pd.read_csv(RUTA_DATOS)
entrena, prueba = (tabla[tabla["particion"] == p] for p in ("entrena", "prueba"))

X_ent = entrena[CARACTERISTICAS].to_numpy(dtype="float32")
X_pru = prueba[CARACTERISTICAS].to_numpy(dtype="float32")
y_ent = entrena["abandona"].to_numpy(dtype="float32")
y_pru = prueba["abandona"].to_numpy(dtype="float32")

media, desv = X_ent.mean(axis=0), X_ent.std(axis=0)
desv[desv == 0] = 1.0
X_ent, X_pru = (X_ent - media) / desv, (X_pru - media) / desv

print(f"entrena {X_ent.shape}   prueba {X_pru.shape}")

In [ ]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(10,)),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer=keras.optimizers.SGD(0.05),
               loss="binary_crossentropy")

historia = modelo.fit(X_ent, y_ent, epochs=300, batch_size=32,
                      validation_data=(X_pru, y_pru), verbose=0)

print("Eso es todo. Cinco llamadas.")
print(f"perdida final entrenamiento {historia.history['loss'][-1]:.4f}")
print(f"perdida final prueba        {historia.history['val_loss'][-1]:.4f}")

Las mismas cifras, aproximadamente, que las sesenta líneas del cuaderno anterior. No
exactamente iguales, y conviene saber por qué: el barajado de los minilotes usa otro
generador, la inicialización de Keras es Glorot y no He, y la aritmética va en `float32`
en vez de `float64`. Ninguna de esas tres diferencias es un error.

### Las cuatro llamadas, leídas una por una

In [ ]:
modelo = keras.Sequential([
    keras.layers.Input(shape=(10,)),                      # forma SIN el eje 0
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),          # bloque 3.3: la fija el problema
])

modelo.compile(
    optimizer=keras.optimizers.Adam(1e-3),                # COMO se actualizan (bloque 5)
    loss="binary_crossentropy",                           # QUE se minimiza (bloque 4)
    metrics=[keras.metrics.AUC(name="auc")],              # QUE se informa, y no se optimiza
)

modelo.summary()

In [ ]:
historia = modelo.fit(
    X_ent, y_ent,
    epochs=60,
    batch_size=32,
    validation_data=(X_pru, y_pru),   # explicito, NO validation_split. Ver mas abajo.
    verbose=0,
)

print("fit devuelve un objeto History, y history.history es un diccionario de listas:")
for clave, valores in historia.history.items():
    print(f"  {clave:10} {len(valores)} valores, del {valores[0]:.4f} al {valores[-1]:.4f}")
print()
print("No hay que guardar nada a mano: ya esta todo ahi.")

In [ ]:
perdida, auc = modelo.evaluate(X_pru, y_pru, verbose=0)
p = modelo.predict(X_pru, verbose=0)

print(f"evaluate  -> perdida {perdida:.4f}   AUC {auc:.4f}")
print(f"predict   -> {p.shape}  {p.dtype}")
print()
print("predict devuelve PROBABILIDADES, no clases:")
print(" ", p[:6].ravel().round(3))
print()
print("Convertirlas en clases es aplicar un umbral, y ese umbral sale del coste")
print("de cada error, no de 0,5. Eso esta entero en la UD4 y aqui se da por sabido.")

### El aviso de `validation_split`

`validation_split=0.2` coge **el último 20 % de las filas, tal cual, sin barajar**. Con los
datos ordenados por clase, la validación puede quedarse con una sola clase; con los datos
ordenados por tiempo —el caso de TechStore— coge los más recientes.

Vamos a verlo, porque es un error que no da ningún aviso.

In [ ]:
# El CSV viene ordenado por cliente, y la particion temporal esta en una columna.
# Si se le pasa validation_split a la tabla entera, esto es lo que se valida:
n = len(X_ent)
corte = int(n * 0.8)
print(f"Con validation_split=0.2 sobre {n} filas de entrenamiento:")
print(f"  entrena con las filas 0 a {corte - 1}")
print(f"  valida  con las filas {corte} a {n - 1}")
print()
print(f"  tasa de abandono en la parte de entrenamiento: {y_ent[:corte].mean():.3f}")
print(f"  tasa de abandono en la parte de validacion:    {y_ent[corte:].mean():.3f}")
print()
print("Aqui la diferencia es tolerable porque el CSV esta ordenado por identificador")
print("de cliente, que no tiene nada que ver con la etiqueta. Pero eso es SUERTE,")
print("no diseño: con los datos ordenados por clase, la validacion se queda con una")
print("sola clase y el entrenamiento parece ir bien mientras la validacion no")
print("significa nada. La regla es pasar validation_data explicito.")

---

## 3. Las tres formas de definir un modelo

`Sequential` para una pila sin bifurcaciones; la API funcional cuando hay varias entradas,
varias salidas o conexiones que saltan capas; subclases cuando la pasada de ida tiene
lógica de verdad. En este módulo se usan las dos primeras.

In [ ]:
# Sequential: una pila.
secuencial = keras.Sequential([
    keras.layers.Input(shape=(10,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
], name="secuencial")

# Funcional: exactamente lo mismo, escrito como un grafo.
entrada = keras.Input(shape=(10,))
h = keras.layers.Dense(16, activation="relu")(entrada)
salida = keras.layers.Dense(1, activation="sigmoid")(h)
funcional = keras.Model(entrada, salida, name="funcional")

print(f"secuencial: {secuencial.count_params()} parametros")
print(f"funcional:  {funcional.count_params()} parametros")
print()
print("Identicos. La API funcional no es 'la avanzada': es la que permite")
print("escribir lo que Sequential no puede.")

In [ ]:
# Lo que Sequential NO puede: una conexion que salta una capa.
# Es el bloque residual del bloque 14, en su version minima.
entrada = keras.Input(shape=(10,))
h = keras.layers.Dense(10, activation="relu")(entrada)
h = keras.layers.Dense(10)(h)
suma = keras.layers.Add()([h, entrada])            # <- esto es lo que Sequential no sabe
h = keras.layers.Activation("relu")(suma)
salida = keras.layers.Dense(1, activation="sigmoid")(h)

residual = keras.Model(entrada, salida, name="residual")
residual.summary()

---

## 4. Lo que aporta Keras, medido

La respuesta perezosa al criterio 2.b es "es más cómodo". La que cuenta es la que se mide.
Esta es la primera medición; la P5.1 las amplía todas.

In [ ]:
LINEAS_NUMPY = 58        # el bucle de entrenamiento del cuaderno 02, sin comentarios
LINEAS_KERAS = 12        # Sequential + compile + fit

comparacion = pd.DataFrame([
    {"dimension": "lineas de codigo", "NumPy": LINEAS_NUMPY, "Keras": LINEAS_KERAS},
    {"dimension": "gradientes", "NumPy": "a mano, y hay que comprobarlos",
     "Keras": "automaticos"},
    {"dimension": "optimizadores", "NumPy": "SGD, y lo que escribas",
     "Keras": "SGD, Adam, RMSprop, AdamW..."},
    {"dimension": "GPU", "NumPy": "no", "Keras": "si, sin cambiar codigo"},
    {"dimension": "retrollamadas", "NumPy": "a mano", "Keras": "integradas"},
    {"dimension": "guardar el modelo", "NumPy": "np.savez, y el codigo aparte",
     "Keras": "un .keras que lo lleva todo"},
    {"dimension": "arranque", "NumPy": "inmediato",
     "Keras": f"{t_import:.1f} s de importacion"},
    {"dimension": "transparencia", "NumPy": "total", "Keras": "hay que fiarse"},
])
print(comparacion.to_string(index=False))

In [ ]:
# El tiempo por epoca, con la MISMA arquitectura y el MISMO tamaño de lote.
# Sin esas dos condiciones la comparacion no vale nada.
def cronometra_keras(n_capas_ocultas, unidades, epocas=30):
    keras.utils.set_random_seed(20262027)
    capas = [keras.layers.Input(shape=(10,))]
    for _ in range(n_capas_ocultas):
        capas.append(keras.layers.Dense(unidades, activation="relu"))
    capas.append(keras.layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(capas)
    m.compile(optimizer="adam", loss="binary_crossentropy")
    m.fit(X_ent[:32], y_ent[:32], epochs=1, batch_size=32, verbose=0)   # calentar
    t0 = time.perf_counter()
    m.fit(X_ent, y_ent, epochs=epocas, batch_size=32, verbose=0)
    return (time.perf_counter() - t0) / epocas, m.count_params()


print(f"{'arquitectura':>22} {'parametros':>11} {'ms/epoca':>10}")
print("-" * 46)
for capas, unidades in [(1, 8), (1, 64), (2, 64), (4, 256)]:
    t, p_ = cronometra_keras(capas, unidades)
    print(f"{f'{capas} x {unidades} unidades':>22} {p_:>11,} {t * 1000:>10.1f}")

Fíjate en lo que dice esa tabla: **multiplicar por cien los parámetros apenas cambia el
tiempo por época**. Con 300 muestras, el coste no está en multiplicar matrices: está en la
sobrecarga fija de recorrer diez lotes por época. Esa es la razón de que en conjuntos
pequeños dé casi igual el tamaño del modelo para el tiempo, y de que lo que sí cambia sea
el sobreajuste.

Es también un aviso metodológico: **una medición hecha en el régimen equivocado da la
conclusión contraria**. Con 60.000 imágenes, esa tabla se ordena al revés.

---

## 5. La ficha de arquitectura

> Criterio **2.c**: *definir el modelo que se quiere implementar según el problema
> planteado*.

En la UD3, la P3.2 pedía la ficha del modelo: pregunta de negocio, unidad de observación,
tipo de tarea, objetivo, características y partición. Esta es la segunda mitad, y se
rellena **antes** de escribir la primera capa.

Lo importante es que **la mitad de las filas no son decisiones**: salen de la tabla del
bloque 3.3 y no hay nada que elegir.

In [ ]:
def ficha_arquitectura(tarea, n_entradas, n_clases=None):
    # Las tres filas que NO se eligen: salen del tipo de tarea.
    reglas = {
        "regresion":      ("Dense(1)",         "ninguna",  "mse"),
        "binaria":        ("Dense(1)",         "sigmoid",  "binary_crossentropy"),
        "multiclase":     (f"Dense({n_clases})", "softmax",
                           "sparse_categorical_crossentropy"),
        "multietiqueta":  (f"Dense({n_clases})", "sigmoid", "binary_crossentropy"),
    }
    salida, activacion, perdida = reglas[tarea]
    return pd.Series({
        "forma de entrada": f"({n_entradas},)",
        "capa de salida": salida,
        "activacion de salida": activacion,
        "perdida": perdida,
    })


for tarea, n_clases in [("binaria", None), ("multiclase", 10),
                        ("regresion", None), ("multietiqueta", 5)]:
    print(f"--- {tarea} ---")
    print(ficha_arquitectura(tarea, 10, n_clases).to_string())
    print()

### La ficha de TechStore, completa

Esta es la ficha que hay que saber rellenar, y es lo que pide la **P5.2**:

| Apartado | Valor | De dónde sale |
|---|---|---|
| Pregunta de negocio | ¿Qué clientes van a dejar de comprar? | UD3, ficha del modelo |
| Unidad de observación | un cliente activo antes del 1/7/2024 | UD3 |
| Tipo de tarea | clasificación binaria | UD3 |
| Forma de entrada | `(10,)` | de `X` |
| Capa de salida | `Dense(1)` | mecánico |
| Activación de salida | `sigmoid` | mecánico |
| Pérdida | `binary_crossentropy` | mecánico |
| Métrica de informe | AUC y F1, **no exactitud** | 24 % de positivos en la población |
| Punto de referencia | recencia > 150 días, F1 0,563, AUC 0,837 | UD3, medido |
| Capacidad inicial | logística, y crecer solo si hace falta | 240 filas |
| Presupuesto | segundos, en CPU | medido arriba |

La fila de la métrica merece un párrafo. El 24 % de la población abandona, así que **un
modelo que conteste siempre "se queda" acierta el 76 %**; en la partición de prueba, que
abandona menos, acierta el 82 %. Esas cifras suenan bien y son la cota mínima, no un
resultado. Comprobémoslo.

In [ ]:
mayoritaria = 1 - y_pru.mean()
print(f"Exactitud del modelo que contesta siempre 'se queda': {mayoritaria:.3f}")
print()

pred = (modelo.predict(X_pru, verbose=0).ravel() >= 0.5).astype(int)
print(f"Exactitud del modelo entrenado:                       "
      f"{(pred == y_pru).mean():.3f}")
print()
print("Si el modelo entrenado no pasa de la primera cifra, no ha aprendido nada")
print("util, y la exactitud no lo dice. Por eso la metrica de informe de esta")
print("ficha es el AUC y el F1, y no la exactitud. Es la leccion de la UD4.")

### La capacidad se elige por el tamaño del conjunto

El orden del bloque 8.2, con un punto de parada en cada paso. Vamos a recorrerlo entero
sobre TechStore y a mirar qué pasa.

In [ ]:
def entrena_y_mide(capas, epocas=100, etiqueta=""):
    keras.utils.set_random_seed(20262027)
    pila = [keras.layers.Input(shape=(10,))]
    for u in capas:
        pila.append(keras.layers.Dense(u, activation="relu"))
    pila.append(keras.layers.Dense(1, activation="sigmoid"))
    m = keras.Sequential(pila)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    m.fit(X_ent, y_ent, epochs=epocas, batch_size=32, verbose=0)
    _, auc_ent = m.evaluate(X_ent, y_ent, verbose=0)
    _, auc_pru = m.evaluate(X_pru, y_pru, verbose=0)
    return {"modelo": etiqueta, "parametros": m.count_params(),
            "AUC entrena": auc_ent, "AUC prueba": auc_pru,
            "hueco": auc_ent - auc_pru}


filas = [entrena_y_mide(capas, etiqueta=etiqueta) for capas, etiqueta in [
    ([],          "logistica (sin capa oculta)"),
    ([8],         "1 capa de 8"),
    ([16, 8],     "2 capas, 16 y 8"),
    ([64, 64],    "2 capas de 64"),
    ([256] * 4,   "4 capas de 256"),
]]

resumen = pd.DataFrame(filas)
print(resumen.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print("El AUC del punto de referencia de la UD3 sobre esta misma particion: 0.837")

> **Ninguna gana al punto de referencia, y el hueco crece con la capacidad.** Ese es el
> resultado del bloque 9, y el cuaderno `UD5_04` lo desarrolla entero. Anótalo.

Y fíjate en la columna `hueco`: **la distancia entre el AUC de entrenamiento y el de
prueba es la medida directa del sobreajuste**, y no hace falta ninguna teoría para verla.
Son dos números.

---

## Ejercicios

### Ejercicio 1. La traducción

Coge este fragmento de código antiguo, tradúcelo a Keras 3 y hazlo funcionar sobre
TechStore:

```python
import tensorflow as tf
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(16, input_dim=10, activation='relu'))
model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
model.compile(optimizer=tf.keras.optimizers.Adam(lr=0.001), loss='binary_crossentropy')
model.fit(X, y, nb_epoch=50)
clases = model.predict_classes(X)
```

Hay **cinco** cosas que hay que cambiar, y dos de ellas dan error y tres no. Di cuáles.

### Ejercicio 2. `validation_split` contra `validation_data`

Ordena `X_ent` e `y_ent` por la etiqueta —todos los ceros primero— y entrena con
`validation_split=0.2`. Dibuja la curva de validación. Después hazlo con
`validation_data` y compara. Explica en una celda de texto qué ha pasado y por qué no
salta ningún aviso.

### Ejercicio 3. La ficha, para otra pregunta

La misma tabla de pedidos de TechStore, otra pregunta: **¿cuánto va a gastar cada cliente
en los próximos 90 días?** Rellena la ficha de arquitectura completa. Cambian el tipo de
tarea, la capa de salida, la activación, la pérdida, la métrica y el punto de referencia.
No hace falta escribir el código: hace falta que la ficha sea coherente.

### Ejercicio 4. El régimen de la medición

Repite la tabla de tiempos del apartado 4 con Fashion-MNIST en lugar de TechStore
(`keras.datasets.fashion_mnist.load_data()`, aplanado a 784 entradas, 10.000 muestras).
¿Se mantiene la conclusión de que el tamaño del modelo apenas afecta al tiempo? Escribe
la conclusión correcta, que tiene que mencionar **en qué régimen** vale cada una.

### Ejercicio 5. El bloque residual

Entrena el modelo `residual` del apartado 3 sobre TechStore y compáralo con el secuencial
del mismo número de capas. ¿Mejora? Con este problema y este tamaño, probablemente no, y
saber decir por qué —el bloque 14.2— vale más que la mejora.

---

## Lo que hay que llevarse de aquí

1. **`import keras`, no `tf.keras`.** Y no mezclar los dos en el mismo fichero.
2. **Cinco llamadas hacen lo que sesenta líneas**: `Sequential`, `compile`, `fit`,
   `evaluate`, `predict`.
3. **`compile` responde a tres preguntas distintas**: cómo se actualiza, qué se minimiza y
   qué se informa. La tercera no se optimiza.
4. **`validation_split` no baraja.** Con datos ordenados es una trampa silenciosa: se pasa
   `validation_data`.
5. **`predict` devuelve probabilidades**, y el umbral sale del coste, no de 0,5.
6. **La API funcional no es la avanzada**: es la que permite escribir lo que `Sequential`
   no puede, empezando por una conexión que salta capas.
7. **Importar Keras tarda segundos**, y con datos pequeños eso domina el coste. Una
   medición hecha en el régimen equivocado da la conclusión contraria.
8. **La mitad de la ficha de arquitectura no son decisiones**: la capa de salida, su
   activación y la pérdida las fija el tipo de problema.
9. **La exactitud de la clase mayoritaria es la cota mínima**, y con un 24 % de positivos
   vale 0,76.
10. **El hueco entre el AUC de entrenamiento y el de prueba mide el sobreajuste**, y crece
    con la capacidad.